"""
# 🤖 AI Tutor Chatbot
## Overview
A conversational AI tutor powered by OpenAI's GPT-4o Mini model,
served through a Gradio web interface with real-time streaming responses.

## Features
- 🔑 Secure API key loading via `.env` file
- 🧠 Multi-turn conversation memory (chat history support)
- ⚡ Streaming responses for a real-time chat experience
- 🌐 Sharable Gradio UI (public link via `share=True`)

## Requirements
    pip install openai python-dotenv gradio

## Setup
1. Create a `.env` file in your project root:
       OPENAI_API_KEY=sk-proj-xxxxxxxxxxxxxxxx
2. Run this script and open the Gradio URL in your browser.
"""

In [3]:
import os                                   # For reading environment variables
from openai import OpenAI                   # OpenAI Python SDK
from dotenv import load_dotenv              # Loads variables from .env file
from IPython.display import display, Markdown  # For rendering Markdown in Jupyter
import gradio as gr                         # Gradio for building the chat UI

In [4]:
# Load environment variables from the .env file into os.environ.
# Must be called BEFORE os.getenv(), otherwise the key won't be found.
load_dotenv()

True

In [5]:
# Retrieve the OpenAI API key from environment variables
openai_api_key = os.getenv("OPENAI_API_KEY")
# Initialise the OpenAI client with the retrieved API key
openai_client = OpenAI(api_key=openai_api_key)

In [7]:
def ai_bot(user_input, history):
    try:
        print(f"User input received:{user_input}")
        print(f"History received:{history}")
        system_prompt = "You are an expert and helpful AI tutor. Explain concepts clearly and concisely."
        message = [{"role": "system", "content": system_prompt}]
        if history:
            for item in history:
                if isinstance(item, dict):
                    message.append({"role": item.get("role"), "content": item.get("content")})
                elif isinstance(item, tuple):
                    usr_msg, bot_msg = item
                    message.append({"role": "user", "content": usr_msg})
                    message.append({"role": "assistant", "content": bot_msg})
        message.append({"role": "user", "content": user_input})

        stream = openai_client.chat.completions.create(model="gpt-4o-mini", messages=message, temperature=0.5, stream=True)
        full_response = ""
        for chunks in stream:
            if chunks.choices[0].delta and chunks.choices[0].delta.content:
                partial_response = chunks.choices[0].delta.content
                full_response = full_response + partial_response
                yield full_response
    except Exception as e:
        print(e)
        print(f"There was an error {e}")


In [8]:
"""
## Gradio ChatInterface Configuration

`gr.ChatInterface` wraps the `ai_bot` generator function and handles:
- Rendering the chat window
- Managing conversation history
- Displaying streaming output in real time
"""

gradio_ui = gr.ChatInterface(
    fn=ai_bot,
    title="🤖 Awesome AI Tutor",
    description="Ask anything to openAI gpt-4o mini LLM model",
    textbox=gr.Textbox(placeholder="Ask the AI....", container=False, scale=7 ),
    flagging_mode="never"
)
print("Launching AI tutor...")
gradio_ui.launch(share=True)

Launching AI tutor...
* Running on local URL:  http://127.0.0.1:7862
* Running on public URL: https://92eecda0002db3955b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


User input received:define python
History received:[]
User input received:define python
History received:[]
